---
title: Calibration and Bragg Disk Detection
authors: [gvarnavides]
date: 2026-08-10
---

In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets

import abtem
import ase
from ase.cluster.cubic import FaceCenteredCubic
import py4DSTEM

abtem.config.set({"dask.lazy": False});

## Sample and Probe

The same gold nanoparticle used in the center-of-mass widget, but illuminated with a
*small* convergence semi-angle. In this nanobeam regime the diffracted disks do not
overlap, which is what makes their positions measurable to sub-pixel precision.

In [ ]:
surfaces = [(1, 0, 0), (1, 1, 0), (1, 1, 1)]
layers = [6, 9, 5]
lc = 4.08
atoms = FaceCenteredCubic('Au', surfaces, layers, latticeconstant=lc)

atoms.cell = [48, 48, 30]
atoms.center()

In [ ]:
energy = 60e3
semiangle_cutoff = 10  # mrad; nanobeam regime, disks do not overlap

potential = abtem.Potential(
    atoms,
    gpts=(192, 192),
    slice_thickness=30,
).build()

probe = abtem.Probe(
    energy=energy,
    semiangle_cutoff=semiangle_cutoff,
).match_grid(
    potential
)

## The Probe Template

The cross-correlation template is the *vacuum probe*: the diffraction pattern recorded
with no sample in the beam. Experimentally this is a real measurement, taken through a
hole in the sample; here we simulate it by propagating the probe through empty space.

We taper the template edge rather than using a hard-edged disk. A soft edge is what
turns the correlation maximum into a well-defined peak instead of a plateau of nearly
equal scores.

In [ ]:
vacuum_dp = probe.build(abtem.CustomScan([[0.0, 0.0]])).diffraction_patterns(max_angle='valid')[0]
template = vacuum_dp.array.copy()
template /= template.max()

# sigmoid-shaped edge, as py4DSTEM's `get_probe_kernel_edge_sigmoid` does
def sigmoid_taper(kernel, inner=0.85, outer=1.15):
    ny, nx = kernel.shape
    qy, qx = np.meshgrid(
        np.fft.fftfreq(ny) * ny,
        np.fft.fftfreq(nx) * nx,
        indexing="ij",
    )
    qr = np.hypot(qy, qx)
    radius = np.sqrt((kernel > 0.5).sum() / np.pi)
    t = (qr - inner * radius) / (outer * radius - inner * radius)
    taper = np.clip(1 - t, 0, 1)
    taper = taper**2 * (3 - 2 * taper)  # smoothstep
    return np.fft.fftshift(taper) * kernel

template = sigmoid_taper(template)
template -= template.mean()  # zero-mean kernel suppresses the flat background
template_FT = np.conj(np.fft.fft2(template))

## Cross-Correlation

The correlation power interpolates between plain cross-correlation (power 1) and phase
correlation (power 0). Cross-correlation is robust to noise but gives broad peaks; phase
correlation gives sharp peaks but amplifies noise. Intermediate values are usually the
right answer.

In [ ]:
def cross_correlate(dp_array, corr_power):
    """Cross/phase correlation of a diffraction pattern against the probe template."""
    m = np.fft.fft2(dp_array) * template_FT
    cc = np.abs(m)**corr_power * np.exp(1j * np.angle(m))
    return np.fft.fftshift(np.real(np.fft.ifft2(cc)))


def detect_disks(cc, min_relative_intensity, min_spacing, subpixel):
    """Locate correlation maxima, i.e. the Bragg disks."""
    peaks = py4DSTEM.process.utils.get_maxima_2D(
        cc,
        subpixel=subpixel,
        minRelativeIntensity=min_relative_intensity,
        minSpacing=min_spacing,
        edgeBoundary=2,
        maxNumPeaks=100,
    )
    return peaks

## Widget

In [ ]:
style = {'description_width': 'initial'}
layout_half = ipywidgets.Layout(width="335px", height="30px")
layout_third = ipywidgets.Layout(width="225px", height="30px")

doses = np.append(np.geomspace(1e2, 1e4, 7), np.inf)

dose_slider = ipywidgets.SelectionSlider(
    options=[(f"{dose:.0f}", i) for i, dose in enumerate(doses)],
    value=7,
    description=r"total dose [$e^-$]",
    style=style,
    layout=layout_half,
)

corr_power_slider = ipywidgets.FloatSlider(
    min=0.0, max=1.0, step=0.05, value=1.0,
    description="correlation power",
    style=style, layout=layout_half,
)

min_intensity_slider = ipywidgets.FloatLogSlider(
    base=10, min=-3, max=0, step=0.05, value=0.05,
    description="min relative intensity",
    style=style, layout=layout_half,
)

min_spacing_slider = ipywidgets.FloatSlider(
    min=1, max=30, step=1, value=10,
    description="min peak spacing [px]",
    style=style, layout=layout_half,
)

subpixel_dropdown = ipywidgets.Dropdown(
    options=['none', 'poly', 'multicorr'],
    value='poly',
    description="subpixel",
    style=style, layout=layout_third,
)

template_toggle = ipywidgets.ToggleButton(
    value=False,
    description="show template",
    style=style, layout=layout_third,
)

In [ ]:
index = [96, 96]

arrays_to_mutate = [
    index,       # probe position, in pixels
    None,        # current diffraction pattern
    None,        # current correlogram
]

In [ ]:
def simulate_dp():
    """Live multislice at the current probe position, with optional shot noise."""
    index_i, index_j = arrays_to_mutate[0]
    pos = np.array([[index_i, index_j]]) * potential.sampling[0]

    exit_wave = probe.multislice(potential, abtem.CustomScan(pos))[0]
    dp = exit_wave.diffraction_patterns(max_angle='valid')

    if dose_slider.value < len(doses) - 1:
        dp = dp.poisson_noise(total_dose=doses[dose_slider.value])

    return dp

In [ ]:
dp = simulate_dp()
arrays_to_mutate[1] = dp

cc = cross_correlate(dp.array, corr_power_slider.value)
arrays_to_mutate[2] = cc

peaks = detect_disks(cc, min_intensity_slider.value, min_spacing_slider.value, subpixel_dropdown.value)

In [ ]:
dpi = 72
with plt.ioff():
    fig, axs = plt.subplots(1, 3, figsize=(675/dpi, 245/dpi), dpi=dpi)

# left panel: sample and probe position
axs[0].patch.set_facecolor('black')
abtem.show_atoms(atoms, scale=1, ax=axs[0], tight_limits=True, title='probe position')
probe_marker, = axs[0].plot(
    [index[1] * potential.sampling[0]],
    [index[0] * potential.sampling[0]],
    marker='+', markersize=12, markeredgewidth=2,
    color=(255/255, 249/255, 148/255),
)

# middle panel: diffraction pattern
im_dp = axs[1].imshow(dp.array**0.25, cmap='gray')
axs[1].set(title="diffraction pattern", xticks=[], yticks=[])

# right panel: correlogram with detected disks
im_cc = axs[2].imshow(cc, cmap='gray')
detected, = axs[2].plot(
    peaks['x'], peaks['y'],
    linestyle='none', marker='o', markersize=7,
    markerfacecolor='none', markeredgewidth=1.5,
    color=(255/255, 249/255, 148/255),
)
axs[2].set(title=f"correlogram: {len(peaks)} disks", xticks=[], yticks=[])

fig.canvas.resizable = False
fig.canvas.header_visible = False
fig.canvas.footer_visible = False
fig.canvas.toolbar_visible = False
fig.canvas.layout.width = '680px'
fig.canvas.toolbar_position = 'bottom'
None

In [ ]:
def update_detection():
    """Re-run correlation and peak finding on the current diffraction pattern."""
    dp = arrays_to_mutate[1]

    if template_toggle.value:
        im_dp.set_data(np.fft.fftshift(template))
        axs[1].set(title="probe template")
    else:
        im_dp.set_data(dp.array**0.25)
        axs[1].set(title="diffraction pattern")
    im_dp.autoscale()

    cc = cross_correlate(dp.array, corr_power_slider.value)
    arrays_to_mutate[2] = cc

    peaks = detect_disks(
        cc,
        min_intensity_slider.value,
        min_spacing_slider.value,
        subpixel_dropdown.value,
    )

    im_cc.set_data(cc)
    im_cc.autoscale()
    detected.set_data(peaks['x'], peaks['y'])
    axs[2].set(title=f"correlogram: {len(peaks)} disks")

    fig.canvas.draw_idle()
    return None


def update_all():
    """Re-simulate the diffraction pattern, then re-detect."""
    arrays_to_mutate[1] = simulate_dp()
    update_detection()
    return None


def onmove(event):
    """ """
    if event.inaxes is not axs[0]:
        return None
    pos = np.array([event.xdata, event.ydata])
    if pos[0] is None:
        return None
    pos = np.mod(np.floor(pos / potential.sampling[0]).astype("int"), potential.gpts).tolist()
    arrays_to_mutate[0] = pos
    probe_marker.set_data(
        [pos[1] * potential.sampling[0]],
        [pos[0] * potential.sampling[0]],
    )
    update_all()
    return None


cid = fig.canvas.mpl_connect('motion_notify_event', onmove)

dose_slider.observe(lambda change: update_all(), "value")

for widget in (corr_power_slider, min_intensity_slider, min_spacing_slider,
               subpixel_dropdown, template_toggle):
    widget.observe(lambda change: update_detection(), "value")

In [ ]:
#| label: app:nanobeam_disk_detection

ipywidgets.VBox(
    [
        ipywidgets.HBox([dose_slider, corr_power_slider]),
        ipywidgets.HBox([min_intensity_slider, min_spacing_slider]),
        ipywidgets.HBox([subpixel_dropdown, template_toggle]),
        fig.canvas
    ],
    layout=ipywidgets.Layout(
        align_items="center"
    )
)